# Clasificación de nubes de puntos 3D: experimentos completos (PC con GPU)

Este cuaderno entrena PointNet y DGCNN en tres experimentos (E1 ModelNet40, E2 ScanObjectNN, E3 entrenar en ModelNet40 y probar en ScanObjectNN), cada uno con 3 semillas y 100 épocas. Son 18 entrenamientos en total.

Corre en mi PC de mesa, y el avance se guarda en la carpeta `results` para poder retomar si se corta. **Cómo se usa:** la primera vez corro todas las celdas de arriba abajo. Si se corta o detengo la celda de entrenamiento, reinicio el kernel, vuelvo a correr las celdas de arriba y la de entrenamiento sigue donde quedó.

## Paso 1. Configuración

Dejo todos los parámetros que se pueden tocar en una sola celda. `SEMILLAS` es el orden en que se hacen las rondas: primero se completan los seis entrenamientos de la semilla 1 y luego los de la 2 y la 3, así que aunque el tiempo no alcance para todo, siempre queda una tabla completa. `MINUTOS_POR_SESION` es el tiempo máximo que entrena la celda antes de pararse sola y guardar el avance (lo subo mucho si quiero dejarlo corriendo de corrido). La precisión mixta (`AMP`) acelera la GPU, pero la dejo apagada en PointNet para que sea igual a las corridas que ya hice con la semilla 1.

In [ ]:
import os
# Parámetros del experimento (si cambio EPOCAS entre sesiones, el scheduler del avance guardado queda desajustado)
SEMILLAS = [1, 2, 3]
EPOCAS = {"pointnet": 100, "dgcnn": 100}
AMP = {"pointnet": 0, "dgcnn": 1}              # 1 = precisión mixta fp16. Si la pérdida da NaN en DGCNN, pongo 0
MINUTOS_POR_SESION = 100000                       # la celda de entrenamiento se detiene sola al cumplirse este tiempo
GUARDAR_CADA = {"pointnet": 5, "dgcnn": 1}     # épocas entre guardados de avance
TAREAS = ["mn40", "sonn", "mn2sonn"]
MODELOS = ["pointnet", "dgcnn"]
DRIVE_OUT = "results"   # aquí se guardan resultados y avances (carpeta local)os.makedirs(DRIVE_OUT, exist_ok=True)


## Paso 2. Verificar la GPU

Sin GPU, DGCNN tardaría demasiado. Si sale `False`, hay que instalar PyTorch con CUDA desde pytorch.org.

In [ ]:
import torch
print('PyTorch', torch.__version__, '| GPU:', torch.cuda.is_available())
!nvidia-smi -L

## Paso 3. Instalar librerías

PyTorch ya debe estar instalado con CUDA. Aquí solo faltan `h5py` (para leer los datasets), `scikit-learn` (métricas) y `matplotlib` (figuras).

In [ ]:
!pip -q install h5py scikit-learn matplotlib

## Paso 4. Escribir el código del proyecto

Las siguientes cuatro celdas crean (o reescriben) los archivos `models.py`, `data.py`, `train.py` y `aggregate.py`. Los separé en archivos para poder lanzar cada entrenamiento como un proceso aparte: si uno falla, no tumba el cuaderno.

`models.py` tiene PointNet (con las T-Nets de entrada y de características) y DGCNN (cuatro capas EdgeConv con k = 20 vecinos). En DGCNN los vecinos se calculan siempre en float32, aunque el resto use precisión mixta, porque en fp16 las distancias pierden resolución y los vecinos cambiarían.

In [ ]:
%%writefile models.py
"""PointNet y DGCNN para clasificación de nubes de puntos (PyTorch)."""
import torch
import torch.nn as nn
import torch.nn.functional as F


class TNet(nn.Module):
    """Red de transformación espacial de PointNet (k x k)."""
    def __init__(self, k):
        super().__init__()
        self.k = k
        self.conv = nn.Sequential(
            nn.Conv1d(k, 64, 1, bias=False), nn.BatchNorm1d(64), nn.ReLU(True),
            nn.Conv1d(64, 128, 1, bias=False), nn.BatchNorm1d(128), nn.ReLU(True),
            nn.Conv1d(128, 1024, 1, bias=False), nn.BatchNorm1d(1024), nn.ReLU(True))
        self.fc = nn.Sequential(
            nn.Linear(1024, 512, bias=False), nn.BatchNorm1d(512), nn.ReLU(True),
            nn.Linear(512, 256, bias=False), nn.BatchNorm1d(256), nn.ReLU(True),
            nn.Linear(256, k * k))
        nn.init.zeros_(self.fc[-1].weight); nn.init.zeros_(self.fc[-1].bias)

    def forward(self, x):                       # x: B x k x N
        f = self.conv(x).max(dim=2)[0]
        m = self.fc(f).view(-1, self.k, self.k)
        return m + torch.eye(self.k, device=x.device)


class PointNet(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.t1, self.t2 = TNet(3), TNet(64)
        self.mlp1 = nn.Sequential(nn.Conv1d(3, 64, 1, bias=False), nn.BatchNorm1d(64), nn.ReLU(True),
                                  nn.Conv1d(64, 64, 1, bias=False), nn.BatchNorm1d(64), nn.ReLU(True))
        self.mlp2 = nn.Sequential(nn.Conv1d(64, 64, 1, bias=False), nn.BatchNorm1d(64), nn.ReLU(True),
                                  nn.Conv1d(64, 128, 1, bias=False), nn.BatchNorm1d(128), nn.ReLU(True),
                                  nn.Conv1d(128, 1024, 1, bias=False), nn.BatchNorm1d(1024), nn.ReLU(True))
        self.head = nn.Sequential(
            nn.Linear(1024, 512, bias=False), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(512, 256, bias=False), nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(256, num_classes))
        self.feat_mat = None

    def forward(self, x):                       # x: B x N x 3
        x = x.transpose(1, 2)
        x = torch.bmm(self.t1(x), x)
        x = self.mlp1(x)
        m = self.t2(x); self.feat_mat = m
        x = torch.bmm(m, x)
        x = self.mlp2(x).max(dim=2)[0]
        return self.head(x)

    def reg_loss(self):
        m = self.feat_mat
        I = torch.eye(m.size(1), device=m.device)
        return torch.mean(torch.norm(torch.bmm(m, m.transpose(1, 2)) - I, dim=(1, 2)))


def knn(x, k):                                   # x: B x C x N
    # Los vecinos se calculan siempre en float32, aunque el resto corra en precisión mixta:
    # con fp16 las distancias pierden resolución y los vecinos cambian.
    with torch.autocast(device_type=x.device.type, enabled=False):
        x = x.float()
        inner = -2 * torch.matmul(x.transpose(2, 1), x)
        xx = (x ** 2).sum(dim=1, keepdim=True)
        return (-xx - inner - xx.transpose(2, 1)).topk(k=k, dim=-1)[1]


def graph_feature(x, k, idx=None):
    B, C, N = x.shape
    if idx is None:
        idx = knn(x, k)
    idx = (idx + torch.arange(B, device=x.device).view(-1, 1, 1) * N).view(-1)
    xt = x.transpose(2, 1).contiguous()
    nb = xt.view(B * N, -1)[idx].view(B, N, k, C)
    xt = xt.view(B, N, 1, C).expand(-1, -1, k, -1)
    return torch.cat((nb - xt, xt), dim=3).permute(0, 3, 1, 2).contiguous()   # B x 2C x N x k


class DGCNN(nn.Module):
    def __init__(self, num_classes, k=20, emb=1024, dropout=0.5):
        super().__init__()
        self.k = k
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 1, bias=False), nn.BatchNorm2d(o), nn.LeakyReLU(0.2))
        self.e1, self.e2, self.e3, self.e4 = blk(6, 64), blk(128, 64), blk(128, 128), blk(256, 256)
        self.conv5 = nn.Sequential(nn.Conv1d(512, emb, 1, bias=False), nn.BatchNorm1d(emb), nn.LeakyReLU(0.2))
        self.head = nn.Sequential(
            nn.Linear(emb * 2, 512, bias=False), nn.BatchNorm1d(512), nn.LeakyReLU(0.2), nn.Dropout(dropout),
            nn.Linear(512, 256, bias=False), nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(dropout),
            nn.Linear(256, num_classes))

    def forward(self, x):                       # x: B x N x 3
        x = x.transpose(1, 2)
        feats = []
        for blk in (self.e1, self.e2, self.e3, self.e4):
            x = blk(graph_feature(x, self.k)).max(dim=-1)[0]
            feats.append(x)
        x = self.conv5(torch.cat(feats, dim=1))
        x = torch.cat((x.max(dim=-1)[0], x.mean(dim=-1)), dim=1)
        return self.head(x)


def build(name, num_classes):
    return {"pointnet": PointNet, "dgcnn": DGCNN}[name](num_classes)

`data.py` lee los HDF5 de ModelNet40 y ScanObjectNN, normaliza cada nube (centrada y dentro de una esfera unitaria), aplica el aumento de datos solo en entrenamiento (escala, traslación y ruido) y define las 10 clases compartidas para E3.

In [ ]:
%%writefile data.py
"""Carga de ModelNet40 (HDF5, 2048 pts) y ScanObjectNN (HDF5, 2048 pts)."""
import glob, os
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset

SONN_CLASSES = ["bag", "bin", "box", "cabinet", "chair", "desk", "display", "door",
                "shelf", "table", "bed", "pillow", "sink", "sofa", "toilet"]
# Clases compartidas entre ModelNet40 y ScanObjectNN (correspondencia elegida en este trabajo)
SHARED = [("bed", "bed"), ("chair", "chair"), ("desk", "desk"), ("door", "door"),
          ("bookshelf", "shelf"), ("monitor", "display"), ("sink", "sink"),
          ("sofa", "sofa"), ("table", "table"), ("toilet", "toilet")]
SHARED_NAMES = [s for _, s in SHARED]

SONN_FILES = {   # variante -> (subcarpeta, archivo base)
    "OBJ_ONLY": ("main_split_nobg", "objectdataset.h5"),
    "OBJ_BG": ("main_split", "objectdataset.h5"),
    "PB_T50_RS": ("main_split", "objectdataset_augmentedrot_scale75.h5"),
}


def _read_h5(paths):
    X, y = [], []
    for p in paths:
        with h5py.File(p, "r") as f:
            X.append(f["data"][:].astype(np.float32)); y.append(f["label"][:].reshape(-1).astype(np.int64))
    return np.concatenate(X), np.concatenate(y)


def modelnet40(root, split):
    """root = carpeta modelnet40_ply_hdf5_2048"""
    paths = sorted(glob.glob(os.path.join(root, f"ply_data_{split}*.h5")))
    assert paths, f"No hay archivos ply_data_{split}*.h5 en {root}"
    names = [l.strip() for l in open(os.path.join(root, "shape_names.txt"))]
    X, y = _read_h5(paths)
    return X, y, names


def scanobjectnn(root, split, variant="PB_T50_RS"):
    """root = carpeta h5_files"""
    sub, base = SONN_FILES[variant]
    pre = "training" if split == "train" else "test"
    p = os.path.join(root, sub, f"{pre}_{base}")
    assert os.path.exists(p), f"No existe {p}"
    X, y = _read_h5([p])
    return X, y, SONN_CLASSES


def restrict_shared(X, y, names, side):
    """Deja solo las 10 clases compartidas y reetiqueta 0..9 segun SHARED_NAMES."""
    col = 0 if side == "mn40" else 1
    keep, new = [], []
    for i, lab in enumerate(y):
        nm = names[lab]
        for j, pair in enumerate(SHARED):
            if pair[col] == nm:
                keep.append(i); new.append(j); break
    keep = np.array(keep)
    return X[keep], np.array(new, dtype=np.int64)


def normalize(X):
    X = X - X.mean(axis=1, keepdims=True)
    return X / np.maximum(np.linalg.norm(X, axis=2).max(axis=1)[:, None, None], 1e-8)


class Clouds(Dataset):
    def __init__(self, X, y, npoints=1024, train=False):
        self.X, self.y, self.n, self.train = normalize(X[:, :, :3]), y, npoints, train

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        pts = self.X[i]
        if self.train:
            pts = pts[np.random.permutation(pts.shape[0])[:self.n]]
            pts = pts * np.random.uniform(2 / 3, 3 / 2, size=(1, 3)).astype(np.float32)
            pts = pts + np.random.uniform(-0.2, 0.2, size=(1, 3)).astype(np.float32)
            pts = pts + np.clip(0.01 * np.random.randn(*pts.shape), -0.05, 0.05).astype(np.float32)
        else:
            pts = pts[:self.n]
        return torch.from_numpy(pts.astype(np.float32)), int(self.y[i])

`train.py` entrena y evalúa un modelo en una tarea con una semilla. Decisiones que importan para que los resultados sean válidos:

- Se reporta el modelo de la **última época**, no el mejor sobre prueba. Elegir la mejor época mirando el conjunto de prueba inflaría los números.
- El avance se guarda de forma atómica (primero a un archivo temporal y luego se renombra), así que el archivo nunca queda a medias si la sesión se corta.
- Guarda también el estado del scheduler y del `GradScaler`, para que al retomar el learning rate continúe donde iba.
- Si se le da un límite de tiempo, termina la época en curso, guarda y sale con código 3.

In [ ]:
%%writefile train.py
"""Entrenamiento y evaluación. Ejemplos:
  python train.py --task mn40  --model dgcnn   --mn40 DATA/modelnet40_ply_hdf5_2048
  python train.py --task sonn  --model pointnet --sonn DATA/h5_files
  python train.py --task mn2sonn --model dgcnn --mn40 ... --sonn ...
Cada corrida se identifica por tarea, modelo y semilla: results/{task}_{model}_s{seed}.json
Si se pasa --stop_after_min, la corrida guarda su avance y sale con código 3 al cumplirse el tiempo.
"""
import argparse, json, os, sys, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, confusion_matrix
import data as D
from models import build

p = argparse.ArgumentParser()
p.add_argument("--task", choices=["mn40", "sonn", "mn2sonn"], required=True)
p.add_argument("--model", choices=["pointnet", "dgcnn"], required=True)
p.add_argument("--mn40"); p.add_argument("--sonn")
p.add_argument("--variant", default="PB_T50_RS")
p.add_argument("--epochs", type=int, default=100)
p.add_argument("--bs", type=int, default=32)
p.add_argument("--npoints", type=int, default=1024)
p.add_argument("--seed", type=int, default=1)
p.add_argument("--out", default="results")
p.add_argument("--max_train", type=int, default=0, help="limita muestras (solo pruebas rápidas)")
p.add_argument("--ckpt", default="", help="ruta de un .pt para guardar y retomar el avance")
p.add_argument("--ckpt_every", type=int, default=1)
p.add_argument("--amp", type=int, default=0, help="1 = precisión mixta fp16 (solo en GPU)")
p.add_argument("--stop_after_min", type=float, default=0, help="0 = sin límite de tiempo")
p.add_argument("--workers", type=int, default=0 if os.name == "nt" else 2, help="procesos del DataLoader; en Windows 0, porque ahí usan spawn y reejecutan este script")
a = p.parse_args()

import random
random.seed(a.seed); torch.manual_seed(a.seed); np.random.seed(a.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(a.seed)
dev = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = bool(a.amp) and dev == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
TAG = f"{a.task}_{a.model}_s{a.seed}"
os.makedirs(a.out, exist_ok=True)

# ---------- datos ----------
if a.task == "mn40":
    Xtr, ytr, names = D.modelnet40(a.mn40, "train"); Xte, yte, _ = D.modelnet40(a.mn40, "test")
    extra = {}
elif a.task == "sonn":
    Xtr, ytr, names = D.scanobjectnn(a.sonn, "train", a.variant); Xte, yte, _ = D.scanobjectnn(a.sonn, "test", a.variant)
    extra = {}
else:  # entrena en ModelNet40 (10 clases compartidas), evalua en ambos dominios
    Xm, ym, nm = D.modelnet40(a.mn40, "train"); Xtr, ytr = D.restrict_shared(Xm, ym, nm, "mn40")
    Xm2, ym2, _ = D.modelnet40(a.mn40, "test"); Xte, yte = D.restrict_shared(Xm2, ym2, nm, "mn40")
    Xs, ys, sn = D.scanobjectnn(a.sonn, "test", a.variant); Xs, ys = D.restrict_shared(Xs, ys, sn, "sonn")
    names = D.SHARED_NAMES
    extra = {"real": (Xs, ys)}
if a.max_train:
    idx = np.random.permutation(len(ytr))[:a.max_train]; Xtr, ytr = Xtr[idx], ytr[idx]
nc = len(names)
tl = DataLoader(D.Clouds(Xtr, ytr, a.npoints, True), a.bs, shuffle=True, drop_last=True, num_workers=a.workers)
vl = DataLoader(D.Clouds(Xte, yte, a.npoints), 64)

# ---------- modelo y optimizador ----------
net = build(a.model, nc).to(dev)
nparams = sum(q.numel() for q in net.parameters())
if a.model == "pointnet":
    opt = torch.optim.Adam(net.parameters(), 1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.StepLR(opt, 20, 0.7)
else:
    opt = torch.optim.SGD(net.parameters(), 0.1, momentum=0.9, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, a.epochs, eta_min=1e-3)


def evaluate(loader):
    net.eval(); P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                out = net(x.to(dev))
            P.append(out.argmax(1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)


def metrics(P, Y):
    cm = confusion_matrix(Y, P, labels=range(nc))
    pc = np.diag(cm) / np.maximum(cm.sum(1), 1)
    return {"oa": float((P == Y).mean()), "macc": float(pc[cm.sum(1) > 0].mean()),
            "f1_macro": float(f1_score(Y, P, average="macro", labels=range(nc), zero_division=0)),
            "per_class_acc": pc.tolist(), "confusion": cm.tolist()}


# ---------- entrenamiento ----------
hist, t0, start = [], time.time(), 0
if a.ckpt and os.path.exists(a.ckpt):   # retoma si hay avance guardado
    ck = torch.load(a.ckpt, map_location=dev, weights_only=False)
    net.load_state_dict(ck["net"]); opt.load_state_dict(ck["opt"]); sch.load_state_dict(ck["sch"])
    if "scaler" in ck: scaler.load_state_dict(ck["scaler"])
    hist, start = ck["hist"], ck["epoch"]; t0 -= ck.get("elapsed", 0)
    print(f"Retomando desde la época {start}", flush=True)
t_session, interrumpido = time.time(), False
for ep in range(start, a.epochs):
    net.train(); tot, n = 0.0, 0
    for x, y in tl:
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            out = net(x)
        loss = F.cross_entropy(out.float(), y, label_smoothing=0.1)
        if a.model == "pointnet": loss = loss + 0.001 * net.reg_loss().float()
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item() * len(y); n += len(y)
    sch.step()
    if not np.isfinite(tot):
        print(f"ERROR {TAG}: la pérdida se volvió NaN en la época {ep+1}. Prueba con --amp 0.", flush=True); sys.exit(4)
    P, Y = evaluate(vl)
    hist.append({"epoch": ep + 1, "loss": tot / n, "test_oa": float((P == Y).mean())})
    print(f"ep {ep+1:3d}/{a.epochs} loss {tot/n:.4f} test_oa {hist[-1]['test_oa']:.4f}", flush=True)
    if a.ckpt and ((ep + 1) % a.ckpt_every == 0 or ep + 1 == a.epochs):
        torch.save({"net": net.state_dict(), "opt": opt.state_dict(), "sch": sch.state_dict(), "scaler": scaler.state_dict(),
                    "hist": hist, "epoch": ep + 1, "elapsed": time.time() - t0}, a.ckpt + ".tmp")
        os.replace(a.ckpt + ".tmp", a.ckpt)   # renombrar es atómico: el archivo nunca queda a medias
    if a.stop_after_min and ep + 1 < a.epochs and (time.time() - t_session) / 60 >= a.stop_after_min:
        if a.ckpt and (ep + 1) % a.ckpt_every != 0:   # asegura que el avance quede guardado
            torch.save({"net": net.state_dict(), "opt": opt.state_dict(), "sch": sch.state_dict(), "scaler": scaler.state_dict(),
                        "hist": hist, "epoch": ep + 1, "elapsed": time.time() - t0}, a.ckpt + ".tmp"); os.replace(a.ckpt + ".tmp", a.ckpt)
        interrumpido = True; break
if interrumpido:
    print(f"PAUSA {TAG}: van {len(hist)}/{a.epochs} épocas. Avance guardado.", flush=True); sys.exit(3)
train_time = time.time() - t0

# ---------- evaluación final (modelo de la ÚLTIMA época, sin elegir la mejor sobre test) ----------
P, Y = evaluate(vl)
res = {"task": a.task, "model": a.model, "args": vars(a), "device": dev, "params_M": nparams / 1e6,
       "train_time_s": train_time, "history": hist, "class_names": names, "test": metrics(P, Y)}
t = time.time(); evaluate(vl); res["ms_per_sample"] = (time.time() - t) / len(yte) * 1000
if "real" in extra:
    Xs, ys = extra["real"]
    Ps, Ys = evaluate(DataLoader(D.Clouds(Xs, ys, a.npoints), 64))
    res["real"] = metrics(Ps, Ys); res["real"]["n"] = int(len(ys))
    np.savez(f"{a.out}/{TAG}_samples.npz", X=D.normalize(Xs[:, :, :3])[:, :a.npoints], y=Ys, p=Ps)
else:
    np.savez(f"{a.out}/{TAG}_samples.npz", X=D.normalize(Xte[:, :, :3])[:, :a.npoints], y=Y, p=P)
res["n_train"], res["n_test"] = int(len(ytr)), int(len(yte))
res["amp"] = use_amp
json.dump(res, open(f"{a.out}/{TAG}.json", "w"), indent=1)
print("OA", res["test"]["oa"], "mAcc", res["test"]["macc"], "F1", res["test"]["f1_macro"])

`aggregate.py` junta las semillas de cada experimento y calcula media y desviación estándar. También genera las figuras: curvas con banda de una desviación, matrices de confusión sumadas entre semillas, exactitud por clase en E3 y ejemplos de aciertos y errores.

In [ ]:
%%writefile aggregate.py
"""Junta todas las semillas: tablas con media y desviación estándar, y figuras.
Uso: python aggregate.py --res results --out figs
Lee results/{task}_{model}_s{seed}.json. Escribe summary.json, summary.csv y las figuras."""
import argparse, glob, json, os, re
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

ap = argparse.ArgumentParser(); ap.add_argument("--res", default="results"); ap.add_argument("--out", default="figs")
a = ap.parse_args(); os.makedirs(a.out, exist_ok=True)
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
SHARED_SONN = ["bed", "chair", "desk", "door", "shelf", "display", "sink", "sofa", "table", "toilet"]
NM = {"pointnet": "PointNet", "dgcnn": "DGCNN"}
TT = {"mn40": "E1: ModelNet40", "sonn": "E2: ScanObjectNN", "mn2sonn": "E3: ModelNet40 → ScanObjectNN"}
COL = {"pointnet": "#8a8a8a", "dgcnn": "#1f4e79"}

R = {}   # (task, model) -> {seed: resultado}
for f in sorted(glob.glob(f"{a.res}/*_s[0-9]*.json")):
    m = re.match(r"(mn40|sonn|mn2sonn)_(pointnet|dgcnn)_s(\d+)\.json$", os.path.basename(f))
    if m: R.setdefault((m[1], m[2]), {})[int(m[3])] = json.load(open(f))
assert R, f"No hay resultados en {a.res}"

def ms(v):
    v = np.array(v, float); return float(v.mean()), (float(v.std(ddof=1)) if len(v) > 1 else float("nan"))

def shared_acc(r):   # E2 evaluado solo sobre las 10 clases compartidas (el modelo igual elige entre 15)
    cm = np.array(r["test"]["confusion"], float); idx = [r["class_names"].index(c) for c in SHARED_SONN]
    return float(np.trace(cm[np.ix_(idx, idx)]) / cm[idx].sum())

S = {}
for (t, m), runs in R.items():
    seeds = sorted(runs); rs = [runs[s] for s in seeds]; d = {"seeds": seeds, "n": len(seeds), "epochs": rs[0]["args"]["epochs"],
        "params_M": rs[0]["params_M"], "amp": rs[0].get("amp", False), "n_train": rs[0]["n_train"], "n_test": rs[0]["n_test"]}
    main = [r["real"] if t == "mn2sonn" else r["test"] for r in rs]            # E3 se reporta sobre ScanObjectNN
    for k in ("oa", "macc", "f1_macro"): d[k] = ms([x[k] for x in main]); d[k + "_por_semilla"] = [x[k] for x in main]
    if t == "mn2sonn":
        d["oa_mn40"] = ms([r["test"]["oa"] for r in rs]); d["macc_mn40"] = ms([r["test"]["macc"] for r in rs])
        d["brecha_pp"] = ms([100 * (r["test"]["oa"] - r["real"]["oa"]) for r in rs])
    if t == "sonn": d["oa_10_clases"] = ms([shared_acc(r) for r in rs])
    d["min_entrenamiento"] = ms([r["train_time_s"] / 60 for r in rs]); d["ms_por_muestra"] = ms([r["ms_per_sample"] for r in rs])
    S[f"{t}.{m}"] = d
json.dump(S, open("summary.json", "w"), indent=1, ensure_ascii=False)

with open("summary.csv", "w", encoding="utf8") as f:
    f.write("experimento,modelo,semillas,epocas,OA_media,OA_std,mAcc_media,mAcc_std,F1_media,F1_std,parametros_M,min_entrenamiento\n")
    for k, d in S.items():
        t, m = k.split(".")
        f.write(f"{t},{m},{d['n']},{d['epochs']},{100*d['oa'][0]:.2f},{100*d['oa'][1]:.2f},{100*d['macc'][0]:.2f},{100*d['macc'][1]:.2f},"
                f"{100*d['f1_macro'][0]:.2f},{100*d['f1_macro'][1]:.2f},{d['params_M']:.2f},{d['min_entrenamiento'][0]:.0f}\n")
print("Resumen (OA %, media ± desv. estándar entre semillas):")
for k, d in S.items():
    print(f"  {k:18s} n={d['n']}  OA {100*d['oa'][0]:.1f} ± {100*d['oa'][1]:.1f}   mAcc {100*d['macc'][0]:.1f}   F1 {100*d['f1_macro'][0]:.1f}"
          + (f"   brecha {d['brecha_pp'][0]:.1f} ± {d['brecha_pp'][1]:.1f} pp" if "brecha_pp" in d else ""))

# 1) curvas: media entre semillas con banda de ± 1 desviación
fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.4))
for i, t in enumerate(TT):
    for m in NM:
        runs = R.get((t, m))
        if not runs: continue
        H = np.array([[h["test_oa"] * 100 for h in r["history"]] for r in runs.values()]); x = np.arange(1, H.shape[1] + 1)
        ax[i].plot(x, H.mean(0), color=COL[m], label=f"{NM[m]} (n={len(H)})", lw=1.4)
        if len(H) > 1: ax[i].fill_between(x, H.mean(0) - H.std(0, ddof=1), H.mean(0) + H.std(0, ddof=1), color=COL[m], alpha=.18, lw=0)
    ax[i].set_title(TT[t], fontsize=8); ax[i].set_xlabel("Época"); ax[i].grid(alpha=.25)
ax[0].set_ylabel("Exactitud en prueba (%)"); ax[0].legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig(f"{a.out}/curvas.png", dpi=220); plt.close(fig)

# 2) matrices de confusión normalizadas por fila, sumando las semillas de cada modelo
for (t, m), runs in R.items():
    rs = list(runs.values()); key = "real" if t == "mn2sonn" else "test"
    cm = sum(np.array(r[key]["confusion"], float) for r in rs); cm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    names = rs[0]["class_names"]; n = len(cm); s = 5.2 if n > 20 else 3.6
    fig, ax = plt.subplots(figsize=(s, s)); im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(n)); ax.set_yticks(range(n)); fs = 5 if n > 20 else 7
    ax.set_xticklabels(names, rotation=90, fontsize=fs); ax.set_yticklabels(names, fontsize=fs)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Clase real"); fig.colorbar(im, fraction=.046)
    ax.set_title(f"{NM[m]}, {TT[t]} ({len(rs)} semilla{'s' if len(rs) > 1 else ''})", fontsize=7.5)
    fig.tight_layout(); fig.savefig(f"{a.out}/confusion_{t}_{m}.png", dpi=220); plt.close(fig)

# 3) exactitud por clase en las 10 clases compartidas: E3 en ModelNet40, E3 en ScanObjectNN
for m in NM:
    runs = R.get(("mn2sonn", m))
    if not runs: continue
    rs = list(runs.values()); names = rs[0]["class_names"]
    pm = np.array([r["test"]["per_class_acc"] for r in rs]) * 100; pr = np.array([r["real"]["per_class_acc"] for r in rs]) * 100
    fig, ax = plt.subplots(figsize=(7.2, 2.6)); x = np.arange(len(names)); w = .38
    ax.bar(x - w / 2, pm.mean(0), w, color="#8a8a8a", label="ModelNet40 (sintético)")
    ax.bar(x + w / 2, pr.mean(0), w, color="#1f4e79", label="ScanObjectNN (real)")
    if len(rs) > 1:
        ax.errorbar(x - w / 2, pm.mean(0), pm.std(0, ddof=1), fmt="none", ecolor="k", lw=.7, capsize=1.5)
        ax.errorbar(x + w / 2, pr.mean(0), pr.std(0, ddof=1), fmt="none", ecolor="k", lw=.7, capsize=1.5)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=30); ax.set_ylabel("Exactitud (%)"); ax.legend(frameon=False, fontsize=7)
    fig.tight_layout(); fig.savefig(f"{a.out}/clases_e3_{m}.png", dpi=220); plt.close(fig)

# 4) ejemplos visuales (semilla más baja de cada corrida): 3 aciertos y 3 errores
for (t, m), runs in R.items():
    s0 = min(runs); f = f"{a.res}/{t}_{m}_s{s0}_samples.npz"
    if not os.path.exists(f): continue
    z = np.load(f); names = runs[s0]["class_names"]; X, y, pr = z["X"], z["y"], z["p"]
    rng = np.random.default_rng(3); ok, bad = np.where(y == pr)[0], np.where(y != pr)[0]
    pick = list(rng.choice(ok, min(3, len(ok)), replace=False)) + list(rng.choice(bad, min(3, len(bad)), replace=False))
    fig = plt.figure(figsize=(7.2, 2.6))
    for j, i in enumerate(pick):
        ax = fig.add_subplot(1, len(pick), j + 1, projection="3d"); P = X[i]; good = y[i] == pr[i]
        ax.scatter(P[:, 0], P[:, 2], P[:, 1], s=1.2, c="#1f4e79" if good else "#b03a2e", alpha=.8)
        ax.set_axis_off(); ax.set_box_aspect((1, 1, 1))
        ax.set_title(f"real: {names[y[i]]}\npred: {names[pr[i]]}", fontsize=6.5, color="#222" if good else "#b03a2e")
    fig.tight_layout(pad=.3); fig.savefig(f"{a.out}/ejemplos_{t}_{m}.png", dpi=220); plt.close(fig)
print("figuras en", a.out)

## Paso 5. Datos

**ScanObjectNN** (variante PB_T50_RS) son los archivos originales de los autores, descargados de su servidor oficial. Verifiqué su hash SHA-256 contra la copia que ya tenía y coinciden.

**ModelNet40** son los modelos CAD originales de Princeton (archivos OFF). Los convertí antes con `convertir_modelnet.py`: de cada modelo se muestrean 2 048 puntos uniformes sobre la superficie y se guardan en HDF5, con el mismo formato que usa `data.py`. El resultado queda en `datos/modelnet40_original_2048`. Esta celda no descarga nada: solo comprueba que las dos carpetas existan.

In [ ]:
import os, glob
MN40 = "datos/modelnet40_original_2048"
SONN = "datos/h5_files"
print("ModelNet40 :", MN40, "| archivos h5:", len(glob.glob(MN40 + "/ply_data_*.h5")), "(esperado 7: 5 de entrenamiento y 2 de prueba)")
print("ScanObjectNN:", SONN, "| existe:", os.path.exists(SONN + "/main_split/training_objectdataset_augmentedrot_scale75.h5"))
assert os.path.exists(MN40 + "/shape_names.txt"), "Falta datos/modelnet40_original_2048: correr antes convertir_modelnet.py"
assert os.path.exists(SONN + "/main_split/test_objectdataset_augmentedrot_scale75.h5"), "Falta ScanObjectNN en datos/h5_files/main_split"

## Paso 6. Reutilizar lo que ya entrené (solo ScanObjectNN)

Antes ya había entrenado PointNet en ScanObjectNN con la semilla 1 y 100 épocas. Si copio ese resultado (`sonn_pointnet.json` y su `_samples.npz`) a la carpeta `results`, esta celda lo registra como la semilla 1, pero **solo si** la configuración coincide exacta (semilla 1, 100 épocas, 1 024 puntos, lote 32, sin precisión mixta).

Los resultados viejos en ModelNet40 (E1 y E3) **no se reutilizan**, porque se entrenaron con otra versión de los datos. Esos se repiten con el ModelNet40 original convertido.

In [ ]:
import json, shutil
for t in ["sonn"]:
    old, new = f"{DRIVE_OUT}/{t}_pointnet.json", f"{DRIVE_OUT}/{t}_pointnet_s1.json"
    if os.path.exists(new) or not os.path.exists(old): continue
    a = json.load(open(old))["args"]
    ok = (a["seed"] == 1 and a["epochs"] == EPOCAS["pointnet"] and a["npoints"] == 1024 and a["bs"] == 32 and not AMP["pointnet"])
    if ok:
        shutil.copy(old, new); shutil.copy(f"{DRIVE_OUT}/{t}_pointnet_samples.npz", f"{DRIVE_OUT}/{t}_pointnet_s1_samples.npz")
        print("Reutilizado:", t, "pointnet semilla 1")
    else:
        print("NO coincide la configuración, se repite:", t, a)

## Paso 7. Prueba rápida

Antes de gastar horas compruebo que todo corre y mido la velocidad. Entreno muy poco (2 épocas de PointNet y 1 época de DGCNN con 640 muestras), una vez en fp32 y otra con precisión mixta. Los números de exactitud de esta celda no significan nada: solo sirven para ver que no hay errores y comparar tiempos. La celda también prueba la pausa y la reanudación: para DGCNN pido que pare tras la primera época y luego lo retomo.

In [ ]:
import subprocess, time, shutil
common = f"--mn40 {MN40} --sonn {SONN} --out prueba"
shutil.rmtree("prueba", ignore_errors=True); shutil.rmtree("prueba_ckpt", ignore_errors=True); os.makedirs("prueba_ckpt")
def sh(cmd):
    t = time.time(); r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-400:], r.stderr[-300:] if r.returncode not in (0, 3) else ""); return r.returncode, time.time() - t
print("PointNet, 2 épocas:"); sh(f"python train.py --task mn40 --model pointnet {common} --epochs 2 --max_train 640")
for amp in (0, 1):
    print(f"DGCNN, 1 época, amp={amp}:"); rc, dt = sh(f"python train.py --task sonn --model dgcnn {common} --epochs 1 --max_train 640 --amp {amp}")
    print(f"   tiempo: {dt:.0f} s (incluye cargar los datos)")
print("Pausa y reanudación de DGCNN:")
rc, _ = sh(f"python train.py --task sonn --model dgcnn {common} --epochs 2 --max_train 640 --amp 1 --ckpt prueba_ckpt/c.pt --stop_after_min 0.001")
print("   código de salida al pausar:", rc, "(esperado 3)")
rc, _ = sh(f"python train.py --task sonn --model dgcnn {common} --epochs 2 --max_train 640 --amp 1 --ckpt prueba_ckpt/c.pt")
print("   código de salida al terminar:", rc, "(esperado 0)")

## Paso 8. Entrenamientos (esta es la celda que repito)

Recorre las 18 combinaciones en orden de semilla. Para cada una: si su resultado ya está en `results`, la salta; si no, entrena con avance guardado en `results`, y al terminar guarda el resultado. Cuando se cumplen los `MINUTOS_POR_SESION`, la corrida en curso termina su época, guarda y la celda se detiene con un mensaje. Para seguir, vuelvo a correr **solo esta celda** (o las de arriba y esta, si reinicié el kernel). No cambio `EPOCAS` entre sesiones.

In [ ]:
import subprocess, time, os, shutil, json
t_ini = time.time(); pendientes = 0; pausado = False
def listo(t, m, s): return os.path.exists(f"{DRIVE_OUT}/{t}_{m}_s{s}.json")
for s in SEMILLAS:
    for m in MODELOS:
        for t in TAREAS:
            if listo(t, m, s): continue
            restante = MINUTOS_POR_SESION - (time.time() - t_ini) / 60
            if restante <= 0.5: pausado = True; pendientes += 1; continue
            ep = EPOCAS[m]; ck = f"{DRIVE_OUT}/ckpt_{t}_{m}_s{s}_e{ep}.pt"
            print(f"\n=== {t} | {m} | semilla {s} | {ep} épocas | quedan {restante:.0f} min de esta sesión ===", flush=True)
            cmd = (f"python -u train.py --task {t} --model {m} --seed {s} --epochs {ep} --amp {AMP[m]} --mn40 {MN40} --sonn {SONN} "
                   f"--out {DRIVE_OUT} --ckpt \"{ck}\" --ckpt_every {GUARDAR_CADA[m]} --stop_after_min {restante:.2f}")
            p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in p.stdout:
                if "Warning" not in line and "warn" not in line: print(line, end="")
            rc = p.wait()
            if rc == 0:
                pass
            elif rc == 3: pausado = True; pendientes += 1
            else: raise RuntimeError(f"Falló {t} {m} semilla {s} (código {rc}). Revisar el mensaje de arriba.")
total = len(SEMILLAS) * len(MODELOS) * len(TAREAS); hechos = sum(listo(t, m, s) for s in SEMILLAS for m in MODELOS for t in TAREAS)
print(f"\nTerminados: {hechos}/{total}")
print("SESIÓN PAUSADA POR TIEMPO. Vuelvo a correr esta misma celda para continuar." if pausado else "ENTRENAMIENTO COMPLETO. Sigo con el Paso 10.")

## Paso 9. Ver qué falta

Una tabla simple con las 18 corridas. `OK` es que ya tiene resultado, y `avance` quiere decir que hay un guardado a medias que se retoma solo.

In [ ]:
print(f"{'semilla':8s}" + "".join(f"{t+'/'+m:18s}" for m in MODELOS for t in TAREAS))
for s in SEMILLAS:
    fila = f"{s:<8d}"
    for m in MODELOS:
        for t in TAREAS:
            est = "OK" if os.path.exists(f"{DRIVE_OUT}/{t}_{m}_s{s}.json") else ("avance" if os.path.exists(f"{DRIVE_OUT}/ckpt_{t}_{m}_s{s}_e{EPOCAS[m]}.pt") else "-")
            fila += f"{est:18s}"
    print(fila)

## Paso 10. Resultados y figuras

Calcula media y desviación estándar entre semillas y genera las figuras. Con una sola semilla la desviación sale `nan`, es normal. Quedan `summary.csv`, `summary.json` y la carpeta `figs`.

In [ ]:
!python aggregate.py --res results --out figs
print('Listo: summary.csv, summary.json y la carpeta figs')